In [2]:
# ✅ Step 1: Imports
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document
from dotenv import load_dotenv
import os

# ✅ Step 2: Load API key
load_dotenv(dotenv_path=".env")
google_api_key = os.getenv("GOOGLE_API_KEY")

# ✅ Step 3: Setup LLM and Embeddings
llm = ChatGoogleGenerativeAI(temperature=0, model="gemini-1.5-flash", google_api_key=google_api_key)
embedding = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", google_api_key=google_api_key)

# ✅ Step 4: Load and split document
with open("sample.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

splitter = CharacterTextSplitter(separator="\n", chunk_size=300, chunk_overlap=50)
texts = splitter.split_text(raw_text)
documents = [Document(page_content=t) for t in texts]

# ✅ Step 5: Vector Store (optional for Retriever chains)
vectorstore = FAISS.from_texts(texts, embedding)
retriever = vectorstore.as_retriever()

print("✅ Base setup complete.")

✅ Base setup complete.


Same pattern as before — StuffDocumentsChain, LLMChain, and langchain.chains are all legacy chain classes that moved out of core langchain into langchain-classic in 1.x. Also worth noting: langchain.prompts should be langchain_core.prompts (you already know this from earlier).

This confirms it: in LangChain v1.0+, langchain.chains no longer exists as a module in the main langchain package — the classic chains (including create_stuff_documents_chain) moved to the separate langchain_classic package, which you already have installed (that's why your original code with langchain_classic.chains.llm.LLMChain worked).

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from dotenv import load_dotenv
import os

load_dotenv()
google_api_key = os.getenv("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(temperature=0, model="gemini-3.5-flash", google_api_key=google_api_key)

documents = [
    Document(page_content="LangChain helps build LLM-powered apps."),
    Document(page_content="Agents use tools to answer questions."),
    Document(page_content="Vector stores enable semantic search."),
]

prompt = ChatPromptTemplate.from_template(
    "Use the following context to answer the question:\n\n{context}\n\nQuestion: {question}"
)

stuff_chain = create_stuff_documents_chain(
    llm,
    prompt,
    document_variable_name="context"
)

response = stuff_chain.invoke({
    "context": documents[:3],
    "question": "What is this document about?"
})

print(response)


Based on the provided context, this document is about **LangChain** (which helps build LLM-powered applications), how **agents** use tools to answer questions, and how **vector stores** enable semantic search.


Also from langchain.prompts import PromptTemplate → from langchain_core.prompts import PromptTemplate (you've got the right one already used elsewhere, just this file still has the old path). Runnable is imported but unused — harmless, but you can drop it.

In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
import os

load_dotenv()
google_api_key = os.getenv("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(temperature=0, model="gemini-3.5-flash", google_api_key=google_api_key)

# Sample docs — replace with your actual documents
documents = [
    Document(page_content="LangChain is a framework for building LLM-powered applications."),
    Document(page_content="It supports chains, agents, memory, and retrieval-augmented generation."),
    Document(page_content="Gemini and OpenAI models can both be plugged in as the underlying LLM."),
]

# Initial prompt to summarize the first chunk
initial_prompt = PromptTemplate.from_template("""
Write a concise summary of the following text:

{context}
""")

# Refine prompt to update the previous summary with new context
refine_prompt = PromptTemplate.from_template("""
We have an existing summary:
"{existing_answer}"

Refine the summary with this new context:
"{context}"

If the context isn't useful, return the original summary.
""")

# Set up individual chains
initial_summary_chain = initial_prompt | llm | StrOutputParser()
refine_summary_chain = refine_prompt | llm | StrOutputParser()

# Start with first chunk
summary = initial_summary_chain.invoke({"context": documents[0].page_content})

# Iteratively refine with remaining docs
for doc in documents[1:]:
    summary = refine_summary_chain.invoke({
        "existing_answer": summary,
        "context": doc.page_content
    })

# Output final summary
print("📄 Refined Summary:\n")
print(summary)

📄 Refined Summary:

"**LangChain** is an LLM application development framework that supports chains, agents, memory, and retrieval-augmented generation, allowing models like Gemini and OpenAI to be plugged in as the underlying LLM."
